# Functions

In [2]:
def generate_reference_style_animation(
    output_dir=".",
    mp4_name="number_categories_reference_v3.mp4",
    gif_name="number_categories_reference_v3.gif",
    preview_name="number_categories_reference_v3.png",
    width_pixels=720,
    height_pixels=1280,
    dpi=100,
    fps=30,
    hold_seconds=0.55,
    transition_seconds=0.55,
    save_mp4=True,
    save_gif=False,
    save_preview=True
):
    from pathlib import Path
    import math

    import numpy as np
    import matplotlib.pyplot as plt

    from matplotlib.animation import (
        FuncAnimation,
        FFMpegWriter,
        PillowWriter
    )

    from matplotlib.colors import to_rgb
    from matplotlib.patches import Rectangle

    # =====================================================
    # FUNÇÕES DE TEORIA DOS NÚMEROS
    # =====================================================
    def is_prime(n):
        if n < 2:
            return False

        if n % 2 == 0:
            return n == 2

        for divisor in range(3, math.isqrt(n) + 1, 2):
            if n % divisor == 0:
                return False

        return True

    def prime_factors(n):
        factors = []
        value = n
        divisor = 2

        while divisor * divisor <= value:
            while value % divisor == 0:
                factors.append(divisor)
                value //= divisor

            divisor += 1 if divisor == 2 else 2

        if value > 1:
            factors.append(value)

        return factors

    def digit_sum(n):
        return sum(int(digit) for digit in str(n))

    def proper_divisor_sum(n):
        if n == 1:
            return 0

        total = 1

        for divisor in range(2, math.isqrt(n) + 1):
            if n % divisor == 0:
                quotient = n // divisor
                total += divisor

                if quotient != divisor:
                    total += quotient

        return total

    def is_semiprime(n):
        return len(prime_factors(n)) == 2

    def is_sphenic(n):
        factors = prime_factors(n)

        return (
            len(factors) == 3
            and len(set(factors)) == 3
        )

    def is_smith(n):
        if n < 4 or is_prime(n):
            return False

        factors = prime_factors(n)

        return digit_sum(n) == sum(
            digit_sum(factor)
            for factor in factors
        )

    def is_happy(n):
        visited = set()
        value = n

        while value != 1 and value not in visited:
            visited.add(value)

            value = sum(
                int(digit) ** 2
                for digit in str(value)
            )

        return value == 1

    def lucky_numbers(limit):
        values = list(range(1, limit + 1, 2))
        index = 1

        while index < len(values):
            step = values[index]

            if step > len(values):
                break

            values = [
                value
                for position, value in enumerate(
                    values,
                    start=1
                )
                if position % step != 0
            ]

            index += 1

        return set(values)

    def is_kaprekar(n):
        if n == 1:
            return True

        square = n * n
        power = 10 ** len(str(n))

        left = square // power
        right = square % power

        return right > 0 and left + right == n

    def polygonal_values(formula, limit):
        values = set()
        index = 1

        while True:
            value = formula(index)

            if value > limit:
                break

            values.add(value)
            index += 1

        return values

    def fibonacci_numbers(limit):
        values = set()
        first = 1
        second = 2

        while first <= limit:
            values.add(first)
            first, second = second, first + second

        return values

    def catalan_numbers(limit):
        values = set()
        catalan = 1
        index = 0

        while catalan <= limit:
            values.add(catalan)

            catalan = (
                catalan
                * 2
                * (2 * index + 1)
                // (index + 2)
            )

            index += 1

        return values

    def pell_numbers(limit):
        values = set()
        previous = 0
        current = 1

        while current <= limit:
            values.add(current)
            previous, current = current, 2 * current + previous

        return values

    # =====================================================
    # CATEGORIAS
    # =====================================================
    def build_categories(limit=100):
        numbers = set(range(1, limit + 1))

        squares = {
            base**2
            for base in range(1, math.isqrt(limit) + 1)
        }

        cubes = {
            base**3
            for base in range(1, 10)
            if base**3 <= limit
        }

        perfect_powers = {
            base**exponent
            for base in range(2, limit + 1)
            for exponent in range(2, 8)
            if base**exponent <= limit
        }

        triangular = polygonal_values(
            lambda n: n * (n + 1) // 2,
            limit
        )

        tetrahedral = polygonal_values(
            lambda n: n * (n + 1) * (n + 2) // 6,
            limit
        )

        square_pyramidal = polygonal_values(
            lambda n: n * (n + 1) * (2 * n + 1) // 6,
            limit
        )

        pentagonal = polygonal_values(
            lambda n: n * (3 * n - 1) // 2,
            limit
        )

        hexagonal = polygonal_values(
            lambda n: n * (2 * n - 1),
            limit
        )

        palette = [
            "#e53935",
            "#ef5350",
            "#f57c00",
            "#fb8c00",
            "#ff9800",
            "#ffb300",
            "#fbc02d",
            "#c0ca33",
            "#7cb342",
            "#43a047",
            "#00a65a",
            "#00b86b",
            "#00a878",
            "#00a896",
            "#00acc1",
            "#0097a7",
            "#039be5",
            "#1e88e5",
            "#3949ab",
            "#5e35b1",
            "#7e57c2",
            "#8e44ad",
            "#ab47bc",
            "#d63384",
            "#e91e63",
            "#f06292",
            "#ec407a",
            "#d81b60",
            "#c2185b",
            "#ad1457",
            "#e64a19",
            "#f4511e",
            "#ff5722",
            "#e53935"
        ]

        category_data = [
            ("Natural Numbers", numbers),

            (
                "Odd Numbers",
                {n for n in numbers if n % 2 == 1}
            ),

            (
                "Even Numbers",
                {n for n in numbers if n % 2 == 0}
            ),

            (
                "Multiples of 3",
                {n for n in numbers if n % 3 == 0}
            ),

            (
                "Multiples of 5",
                {n for n in numbers if n % 5 == 0}
            ),

            (
                "Multiples of 7",
                {n for n in numbers if n % 7 == 0}
            ),

            (
                "Multiples of 10",
                {n for n in numbers if n % 10 == 0}
            ),

            (
                "Prime Numbers",
                {n for n in numbers if is_prime(n)}
            ),

            (
                "Composite Numbers",
                {
                    n
                    for n in numbers
                    if n > 1 and not is_prime(n)
                }
            ),

            (
                "Semiprime Numbers",
                {n for n in numbers if is_semiprime(n)}
            ),

            ("Square Numbers", squares),

            ("Cube Numbers", cubes),

            ("Perfect Powers", perfect_powers),

            ("Triangular Numbers", triangular),

            ("Tetrahedral Numbers", tetrahedral),

            (
                "Square Pyramidal Numbers",
                square_pyramidal
            ),

            ("Pentagonal Numbers", pentagonal),

            ("Hexagonal Numbers", hexagonal),

            (
                "Perfect Numbers",
                {
                    n
                    for n in numbers
                    if n > 1
                    and proper_divisor_sum(n) == n
                }
            ),

            (
                "Abundant Numbers",
                {
                    n
                    for n in numbers
                    if proper_divisor_sum(n) > n
                }
            ),

            (
                "Deficient Numbers",
                {
                    n
                    for n in numbers
                    if proper_divisor_sum(n) < n
                }
            ),

            (
                "Fibonacci Numbers",
                fibonacci_numbers(limit)
            ),

            (
                "Decimal Harshad Numbers",
                {
                    n
                    for n in numbers
                    if n % digit_sum(n) == 0
                }
            ),

            (
                "Sphenic Numbers",
                {n for n in numbers if is_sphenic(n)}
            ),

            (
                "Smith Numbers",
                {n for n in numbers if is_smith(n)}
            ),

            (
                "Binary Palindromic Numbers",
                {
                    n
                    for n in numbers
                    if bin(n)[2:] == bin(n)[2:][::-1]
                }
            ),

            (
                "Decimal Palindromic Numbers",
                {
                    n
                    for n in numbers
                    if str(n) == str(n)[::-1]
                }
            ),

            (
                "Happy Numbers",
                {n for n in numbers if is_happy(n)}
            ),

            (
                "Lucky Numbers",
                lucky_numbers(limit)
            ),

            (
                "Evil Numbers",
                {
                    n
                    for n in numbers
                    if bin(n).count("1") % 2 == 0
                }
            ),

            (
                "Automorphic Numbers",
                {
                    n
                    for n in numbers
                    if str(n * n).endswith(str(n))
                }
            ),

            (
                "Kaprekar Numbers",
                {n for n in numbers if is_kaprekar(n)}
            ),

            (
                "Catalan Numbers",
                catalan_numbers(limit)
            ),

            (
                "Pell Numbers",
                pell_numbers(limit)
            )
        ]

        return [
            {
                "title": title,
                "values": values,
                "color": palette[index]
            }
            for index, (title, values) in enumerate(
                category_data
            )
        ]

    # =====================================================
    # COORDENADAS
    # =====================================================
    def cell_lower_left(number):
        row = (number - 1) // 10
        column = (number - 1) % 10

        return np.array(
            [
                float(column),
                float(9 - row)
            ]
        )

    def ease_in_out(value):
        value = np.clip(value, 0.0, 1.0)

        return (
            value * value
            * (3.0 - 2.0 * value)
        )

    # =====================================================
    # ASSOCIAÇÃO DOS QUADRADOS
    # =====================================================
    def match_positions(old_values, new_values):
        old_values = list(old_values)
        new_values = list(new_values)

        if not old_values or not new_values:
            return []

        old_positions = np.array(
            [cell_lower_left(n) for n in old_values]
        )

        new_positions = np.array(
            [cell_lower_left(n) for n in new_values]
        )

        differences = (
            old_positions[:, None, :]
            - new_positions[None, :, :]
        )

        costs = np.sum(differences**2, axis=2)

        try:
            from scipy.optimize import linear_sum_assignment

            old_indices, new_indices = linear_sum_assignment(
                costs
            )

            return [
                (
                    old_values[old_index],
                    new_values[new_index]
                )
                for old_index, new_index in zip(
                    old_indices,
                    new_indices
                )
            ]

        except ImportError:
            matches = []
            available_old = set(range(len(old_values)))
            available_new = set(range(len(new_values)))

            while available_old and available_new:
                best_pair = None
                best_cost = float("inf")

                for old_index in available_old:
                    for new_index in available_new:
                        cost = costs[old_index, new_index]

                        if cost < best_cost:
                            best_cost = cost
                            best_pair = (
                                old_index,
                                new_index
                            )

                old_index, new_index = best_pair

                matches.append(
                    (
                        old_values[old_index],
                        new_values[new_index]
                    )
                )

                available_old.remove(old_index)
                available_new.remove(new_index)

            return matches

    def create_transition_tracks(
        old_values,
        new_values
    ):
        old_values = set(old_values)
        new_values = set(new_values)

        common = old_values & new_values

        tracks = [
            {
                "start": number,
                "end": number,
                "alpha_start": 1.0,
                "alpha_end": 1.0
            }
            for number in sorted(common)
        ]

        old_remaining = old_values - common
        new_remaining = new_values - common

        matches = match_positions(
            old_remaining,
            new_remaining
        )

        matched_old = {
            old_number
            for old_number, _ in matches
        }

        matched_new = {
            new_number
            for _, new_number in matches
        }

        for old_number, new_number in matches:
            tracks.append(
                {
                    "start": old_number,
                    "end": new_number,
                    "alpha_start": 1.0,
                    "alpha_end": 1.0
                }
            )

        for old_number in sorted(
            old_remaining - matched_old
        ):
            tracks.append(
                {
                    "start": old_number,
                    "end": old_number,
                    "alpha_start": 1.0,
                    "alpha_end": 0.0
                }
            )

        for new_number in sorted(
            new_remaining - matched_new
        ):
            tracks.append(
                {
                    "start": new_number,
                    "end": new_number,
                    "alpha_start": 0.0,
                    "alpha_end": 1.0
                }
            )

        return tracks

    # =====================================================
    # PREPARAÇÃO
    # =====================================================
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    mp4_path = output_path / mp4_name
    gif_path = output_path / gif_name
    preview_path = output_path / preview_name

    categories = build_categories(100)

    transitions = [
        create_transition_tracks(
            categories[index]["values"],
            categories[index + 1]["values"]
        )
        for index in range(len(categories) - 1)
    ]

    maximum_rectangles = max(
        max(
            len(category["values"])
            for category in categories
        ),
        max(
            len(transition)
            for transition in transitions
        )
    )

    hold_frames = max(
        1,
        round(hold_seconds * fps)
    )

    transition_frames = max(
        2,
        round(transition_seconds * fps)
    )

    segment_frames = hold_frames + transition_frames

    total_frames = (
        (len(categories) - 1) * segment_frames
        + hold_frames
    )

    # =====================================================
    # FIGURA
    # =====================================================
    figure_width = width_pixels / dpi
    figure_height = height_pixels / dpi

    fig = plt.figure(
        figsize=(figure_width, figure_height),
        dpi=dpi,
        facecolor="black"
    )

    # A grade ocupa uma região quadrada, como na referência.
    ax = fig.add_axes(
        [0.075, 0.105, 0.85, 0.75]
    )

    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.set_aspect("equal")
    ax.set_facecolor("black")
    ax.axis("off")

    title_old = fig.text(
        0.5,
        0.915,
        "",
        ha="center",
        va="center",
        fontsize=34,
        fontweight="bold",
        fontfamily="DejaVu Serif",
        color="white"
    )

    title_new = fig.text(
        0.5,
        0.915,
        "",
        ha="center",
        va="center",
        fontsize=34,
        fontweight="bold",
        fontfamily="DejaVu Serif",
        color="white",
        alpha=0.0
    )

    # =====================================================
    # RETÂNGULOS REAIS, SEM BORDAS ARREDONDADAS
    # =====================================================
    highlight_rectangles = []

    for _ in range(maximum_rectangles):
        rectangle = Rectangle(
            (0, 0),
            width=1,
            height=1,
            facecolor="#ff8800",
            edgecolor="none",
            linewidth=0,
            alpha=0,
            zorder=1
        )

        ax.add_patch(rectangle)
        highlight_rectangles.append(rectangle)

    # =====================================================
    # LINHAS DA GRADE
    # =====================================================
    for coordinate in range(11):
        ax.plot(
            [coordinate, coordinate],
            [0, 10],
            color="#d8d8d8",
            linewidth=1.35,
            zorder=3
        )

        ax.plot(
            [0, 10],
            [coordinate, coordinate],
            color="#d8d8d8",
            linewidth=1.35,
            zorder=3
        )

    # =====================================================
    # NÚMEROS
    # =====================================================
    number_texts = []

    for number in range(1, 101):
        position = cell_lower_left(number)

        text = ax.text(
            position[0] + 0.5,
            position[1] + 0.5,
            str(number),
            ha="center",
            va="center",
            fontsize=15,
            fontweight="bold",
            fontfamily="DejaVu Serif",
            color="white",
            zorder=4
        )

        number_texts.append(text)

    # =====================================================
    # RENDERIZAÇÃO
    # =====================================================
    def hide_all_rectangles():
        for rectangle in highlight_rectangles:
            rectangle.set_visible(False)
            rectangle.set_alpha(0.0)

    def interpolate_color(color_start, color_end, value):
        rgb_start = np.array(to_rgb(color_start))
        rgb_end = np.array(to_rgb(color_end))

        return (
            (1.0 - value) * rgb_start
            + value * rgb_end
        )

    def render_static_category(category_index):
        hide_all_rectangles()

        category = categories[category_index]

        title_old.set_text(category["title"])
        title_old.set_color(category["color"])
        title_old.set_alpha(1.0)

        title_new.set_alpha(0.0)

        for rectangle, number in zip(
            highlight_rectangles,
            sorted(category["values"])
        ):
            rectangle.set_xy(
                cell_lower_left(number)
            )

            rectangle.set_facecolor(
                category["color"]
            )

            rectangle.set_alpha(1.0)
            rectangle.set_visible(True)

    def render_transition(
        category_index,
        interpolation
    ):
        hide_all_rectangles()

        eased = ease_in_out(interpolation)

        old_category = categories[category_index]
        new_category = categories[category_index + 1]

        title_old.set_text(old_category["title"])
        title_old.set_color(old_category["color"])
        title_old.set_alpha(1.0 - eased)

        title_new.set_text(new_category["title"])
        title_new.set_color(new_category["color"])
        title_new.set_alpha(eased)

        current_color = interpolate_color(
            old_category["color"],
            new_category["color"],
            eased
        )

        for rectangle, track in zip(
            highlight_rectangles,
            transitions[category_index]
        ):
            start_position = cell_lower_left(
                track["start"]
            )

            end_position = cell_lower_left(
                track["end"]
            )

            current_position = (
                (1.0 - eased) * start_position
                + eased * end_position
            )

            current_alpha = (
                (1.0 - eased)
                * track["alpha_start"]
                + eased
                * track["alpha_end"]
            )

            rectangle.set_xy(current_position)
            rectangle.set_facecolor(current_color)
            rectangle.set_alpha(current_alpha)
            rectangle.set_visible(
                current_alpha > 0.001
            )

    def initialize_animation():
        render_static_category(0)

        return [
            title_old,
            title_new,
            *highlight_rectangles,
            *number_texts
        ]

    def update_animation(frame):
        final_category_start = (
            len(categories) - 1
        ) * segment_frames

        if frame >= final_category_start:
            render_static_category(
                len(categories) - 1
            )

            return [
                title_old,
                title_new,
                *highlight_rectangles,
                *number_texts
            ]

        category_index = frame // segment_frames
        local_frame = frame % segment_frames

        if local_frame < hold_frames:
            render_static_category(category_index)

        else:
            interpolation = (
                local_frame - hold_frames
            ) / (transition_frames - 1)

            render_transition(
                category_index,
                interpolation
            )

        return [
            title_old,
            title_new,
            *highlight_rectangles,
            *number_texts
        ]

    animation = FuncAnimation(
        fig,
        update_animation,
        init_func=initialize_animation,
        frames=total_frames,
        interval=1000 / fps,
        blit=False
    )

    generated_files = {}
    errors = {}

    # =====================================================
    # PREVIEW: MULTIPLES OF 5
    # =====================================================
    if save_preview:
        try:
            render_static_category(4)

            fig.savefig(
                preview_path,
                dpi=dpi,
                facecolor="black"
            )

            generated_files["preview"] = str(
                preview_path.resolve()
            )

        except Exception as error:
            errors["preview"] = str(error)

    # =====================================================
    # MP4
    # =====================================================
    if save_mp4:
        try:
            writer = FFMpegWriter(
                fps=fps,
                bitrate=6000,
                codec="libx264",
                extra_args=[
                    "-pix_fmt",
                    "yuv420p"
                ]
            )

            animation.save(
                mp4_path,
                writer=writer,
                dpi=dpi
            )

            generated_files["mp4"] = str(
                mp4_path.resolve()
            )

        except Exception as error:
            errors["mp4"] = str(error)

    # =====================================================
    # GIF
    # =====================================================
    if save_gif:
        try:
            writer = PillowWriter(
                fps=min(fps, 15)
            )

            animation.save(
                gif_path,
                writer=writer,
                dpi=dpi
            )

            generated_files["gif"] = str(
                gif_path.resolve()
            )

        except Exception as error:
            errors["gif"] = str(error)

    plt.close(fig)

    return {
        "generated_files": generated_files,
        "errors": errors,
        "frames": total_frames,
        "duration_seconds": total_frames / fps,
        "resolution": (
            width_pixels,
            height_pixels
        )
    }


def handler():
    result = generate_reference_style_animation(
        output_dir=".",
        mp4_name="number_categories_reference_v3.mp4",
        gif_name="number_categories_reference_v3.gif",
        preview_name="number_categories_reference_v3.png",
        width_pixels=720,
        height_pixels=1280,
        dpi=100,
        fps=30,
        hold_seconds=0.55,
        transition_seconds=0.55,
        save_mp4=True,
        save_gif=False,
        save_preview=True
    )

    print("\nArquivos gerados:")

    for file_type, file_path in result[
        "generated_files"
    ].items():
        print(
            f"{file_type.upper()}: {file_path}"
        )

    if result["errors"]:
        print("\nErros:")

        for file_type, error in result[
            "errors"
        ].items():
            print(
                f"{file_type.upper()}: {error}"
            )

    print(
        f"\nDuração: "
        f"{result['duration_seconds']:.2f} segundos"
    )

    print(
        f"Resolução: "
        f"{result['resolution'][0]}x"
        f"{result['resolution'][1]}"
    )

    return result

# Pipeline

In [4]:
result = handler()


Arquivos gerados:
PREVIEW: C:\Users\Osvaldo\OneDrive\github_oz\number_types\number_categories_reference_v3.png
MP4: C:\Users\Osvaldo\OneDrive\github_oz\number_types\number_categories_reference_v3.mp4

Duração: 35.73 segundos
Resolução: 720x1280


In [ ]:
result = generate_reference_style_animation(
    output_dir=".",
    mp4_name="number_categories_reference_v3.mp4",
    gif_name="number_categories_reference_v3.gif",
    preview_name="number_categories_reference_v3.png",
    width_pixels=720,
    height_pixels=1280,
    dpi=100,
    fps=30,
    hold_seconds=2,
    transition_seconds=2,
    save_mp4=True,
    save_gif=False,
    save_preview=True
)

In [5]:
result = generate_reference_style_animation(
    output_dir="output",
    mp4_name="animacao.mp4",
    gif_name="animacao.gif",
    preview_name="preview.png",

    width_pixels=1080,
    height_pixels=1920,
    fps=30,

    hold_seconds=0.6,
    transition_seconds=0.5,

    save_mp4=True,
    save_gif=False,
    save_preview=True
)